# LoRA fine-tune `qwen2.5:3b-instruct` для furniture-shop tool calling

Работает и на **Kaggle**, и на **Colab** (cell #4 авто-определяет среду).

**Kaggle setup:**
1. Settings → Accelerator → **GPU T4 x2**
2. Settings → Internet → **On**
3. Загрузи `train.jsonl` + `eval.jsonl` как Kaggle Dataset (Add Data → Upload), приатачь к notebook
4. **Save Version → Save & Run All (Commit)** — фоновый запуск, можно закрыть браузер

**Colab setup:**
1. Runtime → Change runtime type → **T4 GPU**
2. Открой левую панель Files → drag-and-drop `train.jsonl` + `eval.jsonl` в корень `/content/`
3. **Runtime → Run all** — НЕ закрывай вкладку (free Colab отключает GPU при неактивности)

**ETA:** ~50-70 минут до финального GGUF.

**На выходе** (Kaggle: `/kaggle/working/`, Colab: `/content/`):
- `qwen2.5-3b-furniture-q8_0.gguf` (~3.3GB) — production GGUF для Ollama
- `Modelfile` — конфиг для `ollama create`
- `qwen2.5-3b-furniture-lora/` (~50MB) — LoRA адаптер для warm-start (incremental retrain)

В Colab — скачай через панель Files (правый клик → Download). В Kaggle — через Output panel.

## 1. Install

In [ ]:
!pip install -q --upgrade unsloth
print('install done')

## 2. Detect environment + locate train.jsonl + eval.jsonl

Auto-detect Kaggle vs Colab. Works in both.

In [ ]:
import os, glob, shutil

# Auto-detect environment: Kaggle has /kaggle/working, Colab has /content
if os.path.isdir('/kaggle/working'):
    ENV = 'kaggle'
    WORK_DIR = '/kaggle/working'
    # Look in attached Kaggle Datasets
    train_candidates = glob.glob('/kaggle/input/**/train.jsonl', recursive=True)
    eval_candidates = glob.glob('/kaggle/input/**/eval.jsonl', recursive=True)
    assert train_candidates, '❌ train.jsonl не найден в /kaggle/input/. Add Data → загрузи Kaggle Dataset с jsonl-файлами.'
    assert eval_candidates, '❌ eval.jsonl не найден в /kaggle/input/.'
    shutil.copy(train_candidates[0], f'{WORK_DIR}/train.jsonl')
    shutil.copy(eval_candidates[0], f'{WORK_DIR}/eval.jsonl')
elif os.path.isdir('/content'):
    ENV = 'colab'
    WORK_DIR = '/content'
    # Files should already be in /content (drag-and-drop in Files panel)
    assert os.path.isfile('/content/train.jsonl'), \
        '❌ /content/train.jsonl не найден. Открой левую панель Files и drag-and-drop train.jsonl + eval.jsonl в /content/'
    assert os.path.isfile('/content/eval.jsonl'), \
        '❌ /content/eval.jsonl не найден. Загрузи через панель Files.'
else:
    raise RuntimeError('Не знаю эту среду. Поддержка: Kaggle, Colab.')

print(f'Environment: {ENV}')
print(f'Work dir: {WORK_DIR}')
print(f'train.jsonl: {sum(1 for _ in open(f"{WORK_DIR}/train.jsonl"))} lines')
print(f'eval.jsonl:  {sum(1 for _ in open(f"{WORK_DIR}/eval.jsonl"))} lines')

os.chdir(WORK_DIR)

## 3. Config

In [ ]:
BASE_MODEL = 'unsloth/Qwen2.5-3B-Instruct'
# 6144 fits augmented admin prompts (~22% of training rows are admin with 10
# tools + injected fake fields; empirical p99≈4900 tokens, max≈5000).
# If T4 OOMs at BATCH=2, drop to BATCH=1 / GRAD_ACCUM=8 (effective batch stays 8).
MAX_SEQ_LEN = 6144

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.0

BATCH = 2
GRAD_ACCUM = 4
EPOCHS = 5  # bumped 3→5: token-to-param ratio 0.17→0.28x, helps rare patterns (multi-step, off-topic)
LR = 2e-4
WARMUP = 20
WEIGHT_DECAY = 0.01
LOG_EVERY = 10

OUTPUT_DIR = f'{WORK_DIR}/outputs'
GGUF_NAME = 'qwen2.5-3b-furniture'

# Q8_0 вместо Q4_K_M — Q4 в прошлом запуске сломала specialized 3B-модель
# (выдавала кашу вместо JSON). Q8 даёт почти zero потерь, файл ~3.3GB.
GGUF_QUANT = 'q8_0'

## 4. Load model + LoRA adapters

`import pandas` ПЕРВЫМ — иначе circular import между unsloth и pandas/datasets.

In [ ]:
import pandas  # noqa: F401
import torch
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
model.print_trainable_parameters()
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')

## 5. Prepare dataset

In [ ]:
from datasets import load_dataset

raw = load_dataset('json', data_files=f'{WORK_DIR}/train.jsonl', split='train')
print(f'train examples: {len(raw)}')

def to_text(example):
    return {'text': tokenizer.apply_chat_template(
        example['messages'], tokenize=False, add_generation_prompt=False)}

train_ds = raw.map(to_text, remove_columns=raw.column_names)
print('rendered first 200 chars:')
print(train_ds[0]['text'][:200])

## 6. Train (~25-35 min on T4)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    args=SFTConfig(
        per_device_train_batch_size=BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        warmup_steps=WARMUP,
        weight_decay=WEIGHT_DECAY,
        logging_steps=LOG_EVERY,
        optim='adamw_8bit',
        lr_scheduler_type='cosine',
        seed=42,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        output_dir=OUTPUT_DIR,
        report_to='none',
        dataset_text_field='text',
        max_seq_length=MAX_SEQ_LEN,
        packing=False,
        save_strategy='epoch',
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part='<|im_start|>user\n',
    response_part='<|im_start|>assistant\n',
)
stats = trainer.train()
print(f'\nTrain done. Loss: {stats.training_loss:.4f}, time: {stats.metrics["train_runtime"]/60:.1f} min')

## 7. Eval — exact match + tool-sequence accuracy

In [ ]:
import json, re
from collections import Counter

FastLanguageModel.for_inference(model)

with open(f'{WORK_DIR}/eval.jsonl') as f:
    eval_lines = [json.loads(l) for l in f]

def extract_plan(text):
    """Извлекает {"plan": [...]} из ответа модели — терпит markdown ```json```, лишний текст."""
    text = text.strip()
    if text.startswith('```'):
        text = re.sub(r'^```\w*\n?', '', text).rstrip('`').strip()
    try:
        return json.loads(text).get('plan')
    except Exception:
        m = re.search(r'\{.*\}', text, re.DOTALL)
        if not m: return None
        try: return json.loads(m.group(0)).get('plan')
        except Exception: return None

def per_field_metrics(pred, exp):
    """Возвращает (tool_match, args_exact, fields_correct, fields_total, missing, extra).
    Без normalize/sort: каждое поле сравнивается строго (включая порядок элементов
    в массивах). args_exact=1 только если ВСЕ поля совпадают точно."""
    if pred is None or exp is None or len(pred) != len(exp):
        return (0, 0, 0, 0, [], [])
    tool_match_all = all(p.get('tool') == e.get('tool') for p, e in zip(pred, exp))
    args_exact_all = True
    fields_correct = fields_total = 0
    missing_fields, extra_fields = [], []
    for p, e in zip(pred, exp):
        p_args = p.get('args') or {}
        e_args = e.get('args') or {}
        all_keys = set(p_args.keys()) | set(e_args.keys())
        for k in all_keys:
            fields_total += 1
            if k not in p_args:
                missing_fields.append(f"{p.get('tool')}.{k}")
                args_exact_all = False
            elif k not in e_args:
                extra_fields.append(f"{p.get('tool')}.{k}")
                args_exact_all = False
            elif p_args[k] != e_args[k]:
                args_exact_all = False
            else:
                fields_correct += 1
    return (int(tool_match_all), int(args_exact_all),
            fields_correct, fields_total, missing_fields, extra_fields)

n = len(eval_lines)
ok_tool = ok_exact = 0
total_correct = total_fields = 0
missing_counter = Counter()
extra_counter = Counter()
fails = []
for i, ex in enumerate(eval_lines):
    msgs = [m for m in ex['messages'] if m['role'] != 'assistant']
    expected = json.loads(ex['messages'][-1]['content'])['plan']
    inp = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors='pt').to('cuda')
    out = model.generate(inp, max_new_tokens=500, do_sample=False, temperature=0.0)
    gen = tokenizer.decode(out[0][inp.shape[1]:], skip_special_tokens=True)
    pred = extract_plan(gen)
    tm, ae, fc, ft, miss, ext = per_field_metrics(pred, expected)
    ok_tool += tm
    ok_exact += ae
    total_correct += fc
    total_fields += ft
    missing_counter.update(miss)
    extra_counter.update(ext)
    if not ae:
        user_q = next(m for m in msgs if m['role'] == 'user')['content']
        fails.append((i, user_q, expected, pred, gen[:200]))

print(f'\n=== EVAL (n={n}) ===')
print(f'Tool-sequence match:    {ok_tool}/{n} = {100*ok_tool/n:.1f}%')
print(f'Args-exact (strict):    {ok_exact}/{n} = {100*ok_exact/n:.1f}%')
if total_fields:
    print(f'Per-field accuracy:     {total_correct}/{total_fields} = {100*total_correct/total_fields:.1f}%')

if missing_counter:
    print(f'\n=== TOP MISSING FIELDS (модель не выдала, а должна была) ===')
    for k, c in missing_counter.most_common(10):
        print(f'  {c}× {k}')
if extra_counter:
    print(f'\n=== TOP EXTRA FIELDS (модель выдала, а не нужно было) ===')
    for k, c in extra_counter.most_common(10):
        print(f'  {c}× {k}')

print(f'\n=== FAILURES (first 15) ===')
for i, q, exp, got, raw in fails[:15]:
    print(f'#{i} {q!r}')
    print(f'  exp: {exp}')
    print(f'  got: {got}')
    if got is None:
        print(f'  raw: {raw!r}')
    print()

## 8. Build llama.cpp (~5 min) — собираем сами потому что Unsloth's auto-install падает

In [ ]:
!apt-get install -y -qq build-essential cmake libcurl4-openssl-dev 2>&1 | tail -3
!rm -rf /root/.unsloth/llama.cpp
!mkdir -p /root/.unsloth
!git clone --depth=1 https://github.com/ggerganov/llama.cpp /root/.unsloth/llama.cpp 2>&1 | tail -3
!cd /root/.unsloth/llama.cpp && cmake -B build -DLLAMA_CURL=OFF -DGGML_CUDA=OFF 2>&1 | tail -5
# -j 2 чтобы избежать OOM при компиляции
!cd /root/.unsloth/llama.cpp && cmake --build build --config Release -j 2 --target llama-quantize 2>&1 | tail -10
!ls -la /root/.unsloth/llama.cpp/build/bin/llama-quantize

## 9. Save LoRA adapter + merge → GGUF f16 → Q8_0

LoRA adapter (~50MB) сохраняется ПЕРВЫМ — он нужен для warm-start (incremental retrain). Затем merge → f16 GGUF → Q8_0 (минимум потерь). `save_pretrained_gguf` обходим — он зависает на собственной apt-get проверке.

In [ ]:
LORA_DIR = f'{WORK_DIR}/{GGUF_NAME}-lora'
MERGED_DIR = f'{WORK_DIR}/{GGUF_NAME}-merged'
F16_GGUF = f'{WORK_DIR}/{GGUF_NAME}-f16.gguf'
FINAL_GGUF = f'{WORK_DIR}/{GGUF_NAME}-{GGUF_QUANT}.gguf'

# 0. Save LoRA adapter (~50MB) — для warm-start будущих iterations
model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)
print(f'\n✓ LoRA adapter saved to {LORA_DIR}')
!ls -lh {LORA_DIR}

# 1. Merge LoRA в base → HF dir
print('\nMerging LoRA into base model (16-bit)...')
model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method='merged_16bit')
print('✓ merged 16-bit saved')

# 2. HF → GGUF f16
print('\nConverting HF → GGUF f16 (~5 min)...')
!pip install -q sentencepiece protobuf
!python /root/.unsloth/llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} --outfile {F16_GGUF} --outtype f16 2>&1 | tail -5
!ls -lh {F16_GGUF}

# 3. f16 → Q8_0 (минимум потерь — Q4 сломала прошлый запуск)
print('\nQuantizing f16 → Q8_0 (~2 min)...')
!/root/.unsloth/llama.cpp/build/bin/llama-quantize {F16_GGUF} {FINAL_GGUF} {GGUF_QUANT} 2>&1 | tail -10
!ls -lh {FINAL_GGUF}

## 10. Modelfile + cleanup промежуточных файлов

Удаляем merged-dir (6GB), f16-GGUF (6GB), training checkpoints. Сохраняем: Q8_0 GGUF, LoRA adapter, Modelfile.

In [ ]:
import os, shutil

modelfile = f'''FROM ./{GGUF_NAME}-{GGUF_QUANT}.gguf

TEMPLATE """{{{{ if .System }}}}<|im_start|>system
{{{{ .System }}}}<|im_end|>
{{{{ end }}}}{{{{ if .Prompt }}}}<|im_start|>user
{{{{ .Prompt }}}}<|im_end|>
{{{{ end }}}}<|im_start|>assistant
{{{{ .Response }}}}<|im_end|>
"""

PARAMETER stop "<|im_end|>"
PARAMETER stop "<|im_start|>"
PARAMETER temperature 0.0
PARAMETER num_ctx 6144
'''
with open(f'{WORK_DIR}/Modelfile', 'w') as f:
    f.write(modelfile)
print('✓ Modelfile written')

# Удаляем большие промежуточные файлы.
# СОХРАНЯЕМ: FINAL_GGUF (Q8_0), LORA_DIR (адаптер), Modelfile.
TRASH = (MERGED_DIR, F16_GGUF, OUTPUT_DIR,
         f'{WORK_DIR}/train.jsonl', f'{WORK_DIR}/eval.jsonl')
for path in TRASH:
    if os.path.isdir(path):
        shutil.rmtree(path)
        print(f'  removed dir: {path}')
    elif os.path.isfile(path):
        os.remove(path)
        print(f'  removed file: {path}')

print(f'\n=== ИТОГ {WORK_DIR}/ ===')
!ls -lh {WORK_DIR}/
print('\n=== Размеры ===')
!du -sh {WORK_DIR}/*

## 11. Что делать после Run All

В выходной папке (`/kaggle/working/` или `/content/`) появятся:
- `qwen2.5-3b-furniture-q8_0.gguf` (~3.3GB) — главное
- `Modelfile` — конфиг
- `qwen2.5-3b-furniture-lora/` (~50MB, папка) — для warm-start

**Скачать:**
- **Kaggle**: правая панель Output → правый клик на каждом файле → Download
- **Colab**: левая панель Files → правый клик → Download. Для LoRA-папки — сначала zip: `!zip -r lora.zip qwen2.5-3b-furniture-lora`, потом скачай zip.

**Локально — заменить старую модель:**

```bash
cd /home/dianakanaeva/PycharmProjects/Diploma/dataset
ollama rm furniture-3b           # удалить предыдущую
ollama create furniture-3b -f Modelfile
ollama run furniture-3b 'Скажи привет'   # smoke-test
docker compose restart backend
```

Затем проверить чат через UI: user-режим (фильтрация / избранное / корзина) и admin-режим (update_stock / update_prices / get_sales_analytics).

**Warm-start для будущих iterations** (когда захочешь докинуть 20-30 примеров):
1. Залей `qwen2.5-3b-furniture-lora/` обратно как Kaggle Dataset (или drag в Colab `/content/`)
2. В этом ноутбуке замени cell 8 (Load model) на:
   ```python
   model, tokenizer = FastLanguageModel.from_pretrained(
       model_name=f'{WORK_DIR}/qwen2.5-3b-furniture-lora',  # или путь к dataset
       max_seq_length=MAX_SEQ_LEN,
       dtype=None,
       load_in_4bit=True,
   )
   model = FastLanguageModel.get_peft_model(model, ...)  # с тем же LORA_R/ALPHA
   ```
3. EPOCHS=1 (не 3 — мы только дообучаем)
4. Run all → ~10 мин вместо 30, без потери знаний.